# 第 11 章 Pandas によるデータ分析: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/11-data-analysis-with-pandas.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
year = 2020                         # 見る年（2011〜2021。2010 は成長率が出ない）
base_prices = [100, 200]            # 物価指数: 基準年の価格（財ごと）
base_quantities = [10, 5]           # 基準年の数量
current_prices = [110, 190]         # 比較年の価格
current_quantities = [8, 6]         # 比較年の数量

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve().parent != _root / "lib":
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 3 面の GDP・成長率・構成比

In [ ]:
from lib.chapter11.economy import (
    expenditure_gdp,
    expenditure_shares,
    growth_rates,
    income_gdp,
    load_economy,
    production_gdp,
    real_gdp,
    recession_years,
    statistical_discrepancy,
)

df = load_economy()
print(f"{year} 年の GDP: 生産 {production_gdp(df)[year]}、"
      f"分配 {income_gdp(df)[year]}、支出 {expenditure_gdp(df)[year]}")
print(f"統計上の不突合: {statistical_discrepancy(df)[year]}")
print(f"実質 GDP: {real_gdp(df)[year]:.1f}、"
      f"成長率: {growth_rates(real_gdp(df))[year]:.2f}%")
print(f"後退した年: {recession_years(df)}")
expenditure_shares(df, year).round(1)

## 物価指数

In [ ]:
from lib.chapter11.economy import laspeyres_index, paasche_index

print(f"ラスパイレス指数: "
      f"{laspeyres_index(base_prices, base_quantities, current_prices):.1f}")
print(f"パーシェ指数: "
      f"{paasche_index(base_prices, current_prices, current_quantities):.1f}")

## 試してみよう

- `year` を 2020 や 2021 にすると、GDP・成長率・構成比が、その年のものに変わります。
  実質成長率がマイナスの年は、2013 と 2020 です（`year` を 2020 にして確かめます）
- `current_prices` を `[130, 190]` にすると、ラスパイレス指数は 102.5 から 112.5 に
  なります。`[110, 150]` にすると、92.5 に下がります
- 基準年と比較年の数量が同じでない（`base_quantities` と `current_quantities` が
  違う）ので、ラスパイレス指数（102.5）とパーシェ指数（101.0）は、少し違います